# CNN baseline — U-Net + ResNet50 on Sen1Floods11 (issue #19)

Scale: 252 hand-labelled training chips, ~31 iterations/epoch at batch 8.

Issue #19 expects water IoU around **0.55–0.65**, below the published figure.
That shortfall is the expected reproduction gap.

## 1. Code and dependencies

In [1]:
import os, shutil

REPO   = "https://github.com/cs59300-deep-learning/nisar-flood-mapping.git"
BRANCH = "feat/19-cnn-baseline"   # "main" once the branch is merged

# Clone outside any previous clone, so re-running this cell replaces it
# instead of nesting a second copy inside the first.
BASE = next(d for d in ["/kaggle/working", "/content", os.getcwd()] if os.path.isdir(d))
os.chdir(BASE)
shutil.rmtree("nisar-flood-mapping", ignore_errors=True)

!git clone -q -b {BRANCH} {REPO}
os.chdir("nisar-flood-mapping")
!git log --oneline -1
!pip install -q segmentation-models-pytorch timm rasterio pyyaml


3ffc8bf (HEAD -> feat/19-cnn-baseline, origin/feat/19-cnn-baseline) feat(E3): refine CNN baseline notebook and fix AMP dtype selection


The scaffold commit created every source file empty, and Python runs an empty module without output or error. The next cell checks the clone actually carries the implementation, so a wrong branch fails here rather than silently producing no results three cells later.


In [2]:
import pathlib

for f in ["src/nisar_flood/cli.py",
          "src/nisar_flood/training/train.py",
          "src/nisar_flood/training/evaluate.py",
          "configs/unet_resnet50.yaml"]:
    size = pathlib.Path(f).stat().st_size
    assert size > 0, f"{f} is empty - the clone is on a branch without the implementation"
    print(f"{size:>7,} bytes  {f}")


  1,868 bytes  src/nisar_flood/cli.py
  9,513 bytes  src/nisar_flood/training/train.py
  3,530 bytes  src/nisar_flood/training/evaluate.py
  1,257 bytes  configs/unet_resnet50.yaml


## 2. Check what hardware you landed on

In [3]:
import torch
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    major = torch.cuda.get_device_capability(0)[0]
    print(f"{name} | {vram:.0f} GB | compute capability {major}.x")
    print("bfloat16 supported:", torch.cuda.is_bf16_supported())
else:
    print("No CUDA. MPS:", torch.backends.mps.is_available(), "- expect a slow run.")

Tesla T4 | 16 GB | compute capability 7.x
bfloat16 supported: True


### Tuning for a larger GPU

The defaults target a 16 GB T4. On an A100 or H100 the code runs unchanged, but
two things are worth knowing:

- **It will not be proportionally faster.** The model is small (32M parameters)
  and the dataset tiny, so on a big GPU the bottleneck moves from compute to
  reading GeoTIFFs off disk. Raise `data.workers` before raising batch size.
- **Resist a much larger batch.** 252 chips at batch 8 is only ~31 optimiser
  steps per epoch. At batch 32 it is 7, which is too few to learn well without
  also changing the learning rate and epoch count — and then the run is no
  longer comparable with the transformer arm (#20).

On Ampere and newer, enabling TF32 is free speed with no accuracy cost here.

In [4]:
# Safe on A100/H100/4090 and ignored elsewhere.
if torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8:
    torch.set_float32_matmul_precision("high")   # TF32 for matmuls
    print("TF32 enabled")

## 3. Data

The download fetches 446 chips plus labels from Google Cloud Storage and takes
a while. If Sen1Floods11 is already mirrored as a dataset on your host, attach
it and point `--root` at that path instead.

In [5]:
!PYTHONPATH=src python -m nisar_flood.cli download --root data/sen1floods11
!PYTHONPATH=src python -m nisar_flood.cli check    --root data/sen1floods11

bolivia: 100%|██████████████████████████████████| 15/15 [00:03<00:00,  4.95it/s]
OK    train    n=252 (expected 252) missing=0 water=9.5%
OK    val      n= 89 (expected 89) missing=0 water=11.0%
OK    test     n= 90 (expected 90) missing=0 water=12.5%
OK    bolivia  n= 15 (expected 15) missing=0 water=15.9%


`check` should print 252 / 89 / 90 / 15 for train / val / test / bolivia, no
missing files, and inputs inside [0, 1]. **Stop here if it does not.**

## 4. Train

In [6]:
!PYTHONPATH=src python -m nisar_flood.cli train --config configs/unet_resnet50.yaml --seed 0

config.json: 100%|██████████████████████████████| 156/156 [00:00<00:00, 530kB/s]

model.safetensors: downloading bytes: ████████████████████▋ | 96.1MB, 7.70MB/s  
model.safetensors: downloading bytes: ██████████████████████| 96.1MB, 9.21MB/s  
model.safetensors: reconstructing file: 100%|██████|  102MB /  102MB, 9.98MB/s  
unet_resnet50 | cuda | float16 | 32,518,114 trainable params | seed 0
train 252 chips, val 89 chips
  epoch   1  train 1.0398  val 1.4065  iou_water_per_chip_mean 0.1683  pooled 0.2184
  epoch   2  train 0.8134  val 1.2033  iou_water_per_chip_mean 0.2553  pooled 0.3793
  epoch   3  train 0.6590  val 0.7720  iou_water_per_chip_mean 0.3553  pooled 0.5638
  epoch   4  train 0.6623  val 0.7416  iou_water_per_chip_mean 0.3746  pooled 0.6032
  epoch   5  train 0.5882  val 0.8204  iou_water_per_chip_mean 0.2981  pooled 0.5597
  epoch   6  train 0.5435  val 0.7304  iou_water_per_chip_mean 0.3738  pooled 0.6221
  epoch   7  train 0.5484  val 0.9953  iou_water_per_chip_mean 0.

Writes to `runs/unet_resnet50/seed0/`: `history.csv`, `curves.png`, `best.pt`,
`last.pt`, `summary.json`. Checkpoint selection uses the validation image-wise
mean water IoU, matching the Sen1Floods11 definition.

## 5. Evaluate on the held-out splits

In [7]:
!PYTHONPATH=src python -m nisar_flood.cli evaluate \
    --checkpoint runs/unet_resnet50/seed0/best.pt

unet_resnet50 | checkpoint epoch 43 | seed 0 | cuda

test:
  iou_water_per_chip_mean    0.4291
  iou_water                  0.6819
  iou_not_water              0.9477
  miou                       0.8148
  precision                  0.8152
  recall                     0.8066
  f1                         0.8109
  n_chips_scored             87

bolivia:
  iou_water_per_chip_mean    0.4633
  iou_water                  0.7046
  iou_not_water              0.9318
  miou                       0.8182
  precision                  0.7781
  recall                     0.8818
  f1                         0.8267
  n_chips_scored             15


written to runs/unet_resnet50/seed0/eval.json


`test` is the headline number for #19. `bolivia` is Sen1Floods11's
out-of-distribution split, reported separately — it is the closest thing in the
source domain to the transfer question E4 asks about NISAR, so it earns a line
in the report.

## 6. Keep the results

In [ ]:
import shutil, pathlib

SEED = 0
SRC  = pathlib.Path(f"runs/unet_resnet50/seed{SEED}")
DEST = pathlib.Path(f"reports/unet_resnet50_seed{SEED}")

# runs/ is gitignored, so the small artefacts are copied into reports/,
# which is tracked. Checkpoints stay behind: *.pt is gitignored too, and
# they belong in the Hugging Face dataset repo (docs/data_sources.md).
DEST.mkdir(parents=True, exist_ok=True)
for name in ["history.csv", "curves.png", "summary.json", "eval.json"]:
    if (SRC / name).exists():
        shutil.copy(SRC / name, DEST / name)

# A single archive at the session root is the simplest thing to download
# from a hosted runtime's file browser.
archive = shutil.make_archive(str(DEST.name), "zip", DEST)
for f in sorted(DEST.iterdir()):
    print(f"{f.stat().st_size:>9,} bytes  {f}")
print(f"\narchive: {archive}")

  117,945 bytes  reports/unet_resnet50_seed0/curves.png
      754 bytes  reports/unet_resnet50_seed0/eval.json
   20,078 bytes  reports/unet_resnet50_seed0/history.csv
      854 bytes  reports/unet_resnet50_seed0/summary.json

archive: /kaggle/working/nisar-flood-mapping/unet_resnet50_seed0.zip


The four files above are #19's acceptance criteria and are tracked in git.
`best.pt` is not: `*.pt` is gitignored and checkpoints belong in the
Hugging Face dataset repo, per `docs/data_sources.md`. Download it from the
session before closing it, or recover it later with
`kaggle kernels output <user>/<kernel>`.